In [27]:
# 새 셀을 만들어서 실행하세요
!pip install langchain_community langchain_experimental langchain_aws requests_aws4auth pypdf

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
# 인덱스 삭제 셀. 실행시키지 말 것
indices_to_delete = [
    "pdf_indexing_smallchunking",
    "pdf_indexing_largechunking"  
]

for index in indices_to_delete:
    url = f"https://vpc-op-an2-hdsteel-poc-ojqio73homvkzwmg4oartqed6u.ap-northeast-2.es.amazonaws.com/{index}"
    response = requests.delete(url, auth=awsauth)
    
    if response.status_code == 200:
        print(f"✅ 성공: {index} 인덱스가 삭제되었습니다.")
    else:
        print(f"❌ 실패: {index} (상태 코드: {response.status_code}, 사유: {response.text})")

✅ 성공: pdf_indexing_simplechunking 인덱스가 삭제되었습니다.


In [3]:
import os, json, boto3, requests
from requests_aws4auth import AWS4Auth
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_aws import BedrockEmbeddings
from langchain_core.documents import Document

# 1. 환경 설정
OPENSEARCH_ENDPOINT = "vpc-op-an2-hdsteel-poc-ojqio73homvkzwmg4oartqed6u.ap-northeast-2.es.amazonaws.com"
REGION = 'ap-northeast-2'
INDEX_NAME = "pdf_test_toc" # 전략 구분을 위해 인덱스명 변경
EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v2:0"
EMBEDDING_DIMENSION = 1024
CHUNK_SIZE = 700
CHUNK_OVERLAP = 100
PDF_PATH = "MX5HEV_2024_ko_KR.pdf"

# 2. 인증 및 모델 설정
session = boto3.Session(region_name=REGION)
bedrock_runtime = session.client(service_name='bedrock-runtime', region_name=REGION)
credentials = session.get_credentials()
awsauth = AWS4Auth(credentials.access_key, credentials.secret_key, REGION, 'es', session_token=credentials.token)
headers = {"Content-Type": "application/json"}
embeddings_model = BedrockEmbeddings(model_id=EMBEDDING_MODEL_ID, region_name=REGION)

print("✅ 셀 1: 환경 설정 완료")

✅ 셀 1: 환경 설정 완료


In [ ]:
def load_and_split_pdf():
    """
    PDF 문서를 로드하고 청크 단위로 분할하여 반환합니다.
    (F-002: 텍스트 추출, 페이지 분리, 청크 분할 및 오버랩)
    """
    if not os.path.exists(PDF_PATH):
        raise FileNotFoundError(f"PDF 파일이 경로에 없습니다: {PDF_PATH}")
        
    print(f"{PDF_PATH} 로드 중")
    

    loader = PyPDFLoader(PDF_PATH)
    pages = loader.load()


    text_splitter = RecursiveCharacterTextSplitter( #추출된 긴 텍스트 작은 조각으로 분할
        chunk_size=CHUNK_SIZE,  #분할된 각 조각이 가질 수 있는 최대 문자 수(토큰 수)
        chunk_overlap=CHUNK_OVERLAP, #인접한 두 청크 사이 중복되는 문자 수. 문맥 손실 방지
        separators=["\n\n", "\n", " ", ""] #텍스트 쪼갤 때 시도할 구분 기호들의 우선순위
    )
#최우선순위 "\n\n"로 쪼갠 후, 700자 보다 작으면 최종 청크, 크면 그 순위로("\n") 또 쪼갬

    docs = text_splitter.split_documents(pages)

    print(f"원본 페이지 수: {len(pages)}")
    print(f"분할된 청크 수: {len(docs)}")
    
    return docs
    
if __name__ == '__main__':
    try:
        chunks = load_and_split_pdf()
        if chunks:
            print("\n--- 첫 번째 청크 미리보기 ---")
            print(f"내용 (일부): {chunks[0].page_content[:200]}...")
            print(f"메타데이터: {chunks[0].metadata}")
    except FileNotFoundError as e:
        print(f"오류: {e}")

✅ 셀 2: PDF 로드 및 7개 섹션 매핑 완료


In [ ]:
INDEX_BODY = {
    "settings": {
        "index": {
            "knn": True #knn:K-Nearest Neighbors.
        }
    },
    "mappings": {
        "properties": {
            "vector_field": {
                "type": "knn_vector",
                "dimension": EMBEDDING_DIMENSION,
                "method": {
                    "name": "hnsw",
                    "space_type": "l2",
                    "engine": "nmslib"
                }
            },
            "text": {"type": "text"},
            "source": {"type": "keyword"},
            "page_number": {"type": "integer"}
        }
    }
}

# ----------------------------------------------------
# 인덱싱 기능 함수
# ----------------------------------------------------

def create_opensearch_index():
    """
    OpenSearch 인덱스를 생성하거나 이미 존재하면 확인합니다.
    """
    url = f"https://{OPENSEARCH_ENDPOINT}/{INDEX_NAME}"
    try:
        response = requests.put(url, auth=awsauth, headers=headers, data=json.dumps(INDEX_BODY))
        response.raise_for_status()
        print(f"인덱스 '{INDEX_NAME}' 생성 완료.")
    except requests.exceptions.HTTPError as e:
        if response.status_code == 400 and "already exists" in response.text:
            print(f"인덱스 '{INDEX_NAME}' 이미 존재함. 재사용합니다.")
        else:
            print(f"인덱스 생성 실패: HTTP {response.status_code} - {response.text}")
            raise e

def index_documents(docs):
    """
    분할된 문서를 벡터화하여 OpenSearch에 벌크 인덱싱합니다. (F-001)
    """
    create_opensearch_index() # 인덱스 생성 선행

    print(f"{len(docs)}개 청크 임베딩 및 인덱싱 시작...")
    
    bulk_data = []
    
    # LangChain의 get_embeddings() 함수를 사용하면 여러 청크를 한 번에 처리하여 효율적입니다.
    texts = [doc.page_content for doc in docs]
    vectors = embeddings_model.embed_documents(texts) 
    
    for i, (doc, vector) in enumerate(zip(docs, vectors)):
        # 1) 메타데이터 (인덱스 요청)
        bulk_data.append(json.dumps({
            "index": {
                "_index": INDEX_NAME, 
                "_id": f"pdf_chunk_{i}"
            }
        }))
        
        # 2) 문서 데이터
        # OpenSearch 인덱스 스키마에 맞게 각 청크 정보 구조화 한 것
        metadata = doc.metadata
        bulk_data.append(json.dumps({
            "text": doc.page_content, #청크의 원문 텍스트
            "vector_field": vector, #Bedrock에서 생성된 1024차원 벡터
            "source": metadata.get('source', 'Unknown'), #PDF 파일명
            "page_number": metadata.get('page', -1) #페이지 번호
        }))

    bulk_payload = "\n".join(bulk_data) + "\n"
    bulk_url = f"https://{OPENSEARCH_ENDPOINT}/_bulk"

    try:
        bulk_response = requests.post(
            bulk_url, 
            auth=awsauth, 
            headers={"Content-Type": "application/x-ndjson"},
            data=bulk_payload.encode('utf-8')
        )
        bulk_response.raise_for_status()
        
        # 에러 체크
        response_json = bulk_response.json()
        if response_json.get('errors'):
            print("일부 문서 인덱싱 중 오류가 발생했습니다.")
            # 오류 내용 자세히 출력
            # for item in response_json['items']:
            #     if item['index'].get('error'):
            #         print(f"오류: {item['index']['error']}")

        print(f"총 {len(docs)}개 청크 인덱싱 완료")

    except requests.exceptions.RequestException as e:
        print(f"벌크 인덱싱 실패: {e}")
        raise e
    
    refresh_url = f"https://{OPENSEARCH_ENDPOINT}/{INDEX_NAME}/_refresh"
    refresh_response = requests.post(refresh_url, auth=awsauth, headers={"Content-Type": "application/json"})
    refresh_response.raise_for_status()
    print("인덱스 Refresh 완료. 검색 준비됨.")

# 이 파일 단독 실행 시 인덱스 생성만 테스트
if __name__ == '__main__':
    try:
        create_opensearch_index()
    except Exception as e:
        print(f"테스트 실패: {e}")

✅ 셀 3: 메타데이터 주입 완료 (총 7개 청크)
샘플 메타데이터: {'producer': 'Antenna House PDF Output Library 7.2.1700', 'creator': 'AH Formatter V7.2 R1 for Windows (x64) : 7.2.1.53371 (2021-09-14T12:45+09)', 'creationdate': '2025-12-10T16:19:25+09:00', 'author': 'UDT 25.11.12', 'moddate': '2025-12-10T16:19:25+09:00', 'keywords': 'ko_KR,hyundai,MX5HEV,2024,engine,1.14,Hyundai_MX5_HEV', 'title': 'Document', 'subject': 'Hyundai_MX5_HEV', 'trapped': '/False', 'source': 'MX5HEV_2024_ko_KR.pdf', 'total_pages': 653, 'page': 458, 'page_label': '459', 'section_name': '후측방 모니터(BVM)'}


In [ ]:
def vector_search(query_text, k=5):
    """
    쿼리 텍스트를 임베딩하여 OpenSearch에서 k-NN 검색을 수행합니다. (F-005)
    """
    print(f"\n벡터 검색 시작: '{query_text}'")
    
    # 1. 쿼리 텍스트 임베딩
    query_vector = embeddings_model.embed_query(query_text)

    # 2. k-NN 검색 쿼리
    knn_query = {
        "size": k,
        "query": {
            "knn": {
                "vector_field": {
                    "vector": query_vector,
                    "k": k
                }
            }
        },
        "_source": ["text", "source", "page_number"] # F-006 관련 정보
    }

    url = f"https://{OPENSEARCH_ENDPOINT}/{INDEX_NAME}/_search"
    response = requests.post(url, auth=awsauth, headers=headers, data=json.dumps(knn_query))
    response.raise_for_status()
    
    return parse_search_results(response.json())


def hybrid_search(query_text, k=5):
    """
    k-NN과 match 쿼리를 결합합니다.
    """
    print(f"\n🔬 하이브리드 검색 시작 (벡터+키워드): '{query_text}'")
    
    # 1. 쿼리 텍스트 임베딩
    query_vector = embeddings_model.embed_query(query_text)
    
    # 2. OpenSearch의 bool 쿼리를 사용한 단순 하이브리드 쿼리
    hybrid_query = {
        "size": k,
        "query": {
            "bool": {
                # 1. 키워드 검색 (match)
                "should": [
                    {
                        "match": {
                            "text": {
                                "query": query_text,
                                "boost": 2 # 키워드 일치 시 가중치 부여
                            }
                        }
                    },
                    # 2. 벡터 검색 (knn) - bool 쿼리 내에서 knn은 OpenSearch 2.1 이상에서 지원
                    {
                        "knn": {
                            "vector_field": {
                                "vector": query_vector,
                                "k": k,
                                "boost": 1 # 벡터 유사성 가중치
                            }
                        }
                    }
                ],
                "minimum_should_match": 1 # 둘 중 하나만 일치해도 결과 반환
            }
        },
        "_source": ["text", "source", "page_number"]
    }
    
    url = f"https://{OPENSEARCH_ENDPOINT}/{INDEX_NAME}/_search"
    response = requests.post(url, auth=awsauth, headers=headers, data=json.dumps(hybrid_query))
    response.raise_for_status() # 400 에러 재확인
    
    return parse_search_results(response.json())

def parse_search_results(search_response):
    """
    검색 결과를 파싱하여 출력 포맷에 맞춥니다. (F-006)
    """
    results = []
    
    for hit in search_response.get('hits', {}).get('hits', []):
        score = hit.get('_score', 0.0)
        source = hit.get('_source', {})
        
        # 하이라이트된 텍스트 스니펫 
        # 실제 하이라이팅은 OpenSearch 쿼리에 highlight 옵션을 추가해야 하지만, 
        # 여기서는 간단히 텍스트의 일부를 스니펫으로 간주합니다.
        snippet = source.get('text', 'N/A')[:150] + "..." 
        #OpenSearch에서 검색된 전체 텍스트 청크(source.get('text')) 중에서, 앞에서부터
        #150자만 잘라내서 ...을 붙여서 만든 요약 텍스트
        #목적: 전체 텍스트 보여주는 대신, 핵심 내용의 첫 부분만 사용자에게 빠르게 보여주기
        
        
        results.append({
            "score": score,
            "text": source.get('text'),
            "snippet": snippet,
            "source": source.get('source'),
            "page_number": source.get('page_number')
        })
        
    return results

def display_results(results, search_type):
    """
    검색 결과를 사용자 친화적으로 출력합니다.
    """
    print(f"\n--- {search_type} 검색 결과 ({len(results)}건) ---")
    if not results:
        print("검색된 문서가 없습니다.")
        return

    for i, res in enumerate(results):
        print(f"[{i+1}] 관련도 점수: {res['score']:.4f} (F-006)")
        print(f"    출처: {res['source']} | 페이지: {res['page_number']} (F-006)")
        print(f"    스니펫: {res['snippet']}")
        print("-" * 50)
    
# 이 파일은 main.py에서 임포트하여 사용

인덱스 생성 결과: 200
✅ 셀 4: 목차 기반 인덱싱 완료


In [ ]:
# [Cell 5] 검색 테스트 실행
test_query = "후측방 모니터(BVM)은 어떤 기능을 하는지 설명해 주세요"

print("==================================================")
print(f"🔍 테스트 쿼리: {test_query}")
print("==================================================")

# 1. 벡터 검색 (시맨틱 검색) 실행 및 출력
try:
    # k=3으로 설정하여 1, 2, 3순위만 가져옵니다.
    v_results = vector_search(test_query, k=3)
    display_results(v_results, "벡터 검색 (시맨틱)")
except Exception as e:
    print(f"❌ 벡터 검색 실패: {e}")

print("\n" + "="*50)

# 2. 하이브리드 검색 (벡터 + 키워드) 실행 및 출력
try:
    h_results = hybrid_search(test_query, k=3)
    display_results(h_results, "하이브리드 검색")
except Exception as e:
    print(f"❌ 하이브리드 검색 실패: {e}")


🔍 Hybrid 검색 시작: '후측방 모니터(BVM) 기능에 대해 알려주세요'


NameError: name 'embeddings_model' is not defined

In [ ]:
# [Cell 6] 이미 구축된 인덱스를 활용한 RAG 챗봇 구현
from langchain_aws import ChatBedrock
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

CHAT_INDEX = "pdf_indexing_agentchunking"

chat_llm = ChatBedrock(
    model_id="anthropic.claude-3-5-sonnet-20240620-v1:0",
    model_kwargs={"temperature": 0.1}, # 약간의 유연함
    region_name=REGION
)

def ask_manual_chatbot(question):
    print(f"질문: {question}")
    
    # 관련 정보 검색 (하이브리드 검색 활용)
    # 기존에 정의된 embeddings_model과 awsauth 등을 그대로 사용
    query_vector = embeddings_model.embed_query(question)
    
    search_query = {
        "size": 3,
        "query": {
            "bool": {
                "should": [
                    {"match": {"text": {"query": question, "boost": 10}}},
                    {"knn": {"vector_field": {"vector": query_vector, "k": 3, "boost": 2}}}
                ]
            }
        }
    }
    
    url = f"https://{OPENSEARCH_ENDPOINT}/{CHAT_INDEX}/_search"
    res = requests.post(url, auth=awsauth, headers=headers, data=json.dumps(search_query)).json()
    
    # 검색된 결과 정리
    context_list = []
    source_info = []
    for hit in res.get('hits', {}).get('hits', []):
        context_list.append(hit['_source']['text'])
        source_info.append(f"{hit['_source']['page_number']}페이지")
    
    context = "\n\n".join(context_list)
    
    # 프롬프트 구성
    prompt = ChatPromptTemplate.from_template("""
    당신은 현대 차량 매뉴얼을 잘 아는 전문 상담원입니다. 
    아래 제공된 [매뉴얼 내용]을 바탕으로 사용자의 질문에 친절하게 답변하세요.
    
    지침:
    1. 반드시 제공된 [매뉴얼 내용]에 있는 정보만 사용하여 답변하세요.
    2. 만약 내용에 답이 없다면 "죄송합니다. 해당 내용은 매뉴얼에서 찾을 수 없습니다."라고 답하세요.
    3. 답변 끝에 참고한 페이지 번호를 적어주세요.

    [매뉴얼 내용]:
    {context}

    질문: {question}
    """)

    # 체인 실행 및 답변 생성
    chain = prompt | chat_llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    
    print("-" * 50)
    print(f"챗봇 답변:\n{answer}")
    print(f"참고 출처: {', '.join(set(source_info))}")
    print("-" * 50)

# 테스트 실행
#ask_manual_chatbot("후측방 모니터(BVM)를 켜는 방법이 뭐야?")

In [ ]:
# 사용자 입력을 받는 실시간 대화 인터페이스
def start_chat():
    print("현대 차량 매뉴얼 챗봇입니다. 무엇이든 물어보세요!")
    print("(종료하려면 'exit', 'quit', '종료'를 입력하세요.)")
    print("-" * 50)
    
    while True:
        # 사용자로부터 질문 입력 받기
        user_input = input("\n질문을 입력하세요: ").strip()
        
        # 종료 조건 확인
        if user_input.lower() in ['exit', 'quit', '종료', 'q']:
            print("챗봇을 종료합니다. 감사합니다!")
            break
            
        if not user_input:
            print("질문을 입력해 주세요.")
            continue
            
        # 기존에 만든 ask_manual_chatbot 함수 호출
        try:
            ask_manual_chatbot(user_input)
        except Exception as e:
            print(f"오류가 발생했습니다: {e}")

# 대화 시작
start_chat()

🚗 현대제철 차량 매뉴얼 챗봇입니다. 무엇이든 물어보세요!
(종료하려면 'exit', 'quit', '종료'를 입력하세요.)
--------------------------------------------------
🤔 질문: EPB 경고등은 언제 켜져?
--------------------------------------------------
🤖 챗봇 답변:
EPB 경고등은 다음과 같은 경우에 켜집니다:

1. 시동을 'ON' 하면 켜지고 약 3초 후 꺼집니다.

2. 3초가 지나도 경고등이 꺼지지 않거나 주행 중 경고등이 켜지면 전자식 파킹 브레이크에 이상이 있는 것입니다.

3. EPB 스위치를 비정상적으로 작동하였을 경우(과도한 조작, 지속적 작동)에도 EPB 경고등이 켜질 수 있습니다.

이러한 상황에서 EPB 경고등이 계속 켜져 있다면, 당사 직영 하이테크센터나 블루핸즈에서 점검을 받으셔야 합니다.

(참고 페이지: 134, 135)
📍 참고 출처: 139페이지, 22페이지, 144페이지
--------------------------------------------------
챗봇을 종료합니다. 감사합니다!
